In [5]:
!pip install gensim

In [6]:
import numpy as np
import nltk
from nltk.tokenize import word_tokenize
import gensim.downloader as api
from sklearn.metrics.pairwise import cosine_similarity

In [7]:

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [8]:
def preprocess(text):
    tokens = word_tokenize(text.lower())

    return [word for word in tokens if word.isalpha()]

In [9]:
def get_average_embedding(sentence, model):
    tokens = preprocess(sentence)
    word_vectors = []


    for word in tokens:
        if word in model:
            word_vectors.append(model[word])

    if len(word_vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(word_vectors, axis=0)

In [10]:
import gensim.downloader as api

info = api.info()

print("Available models:")
for model_name, model_info in info['models'].items():
    print(f"- {model_name} (size: {model_info.get('file_size', 'N/A')} bytes, description: {model_info.get('description', 'No description available')})")

Available models:
- fasttext-wiki-news-subwords-300 (size: 1005007116 bytes, description: 1 million word vectors trained on Wikipedia 2017, UMBC webbase corpus and statmt.org news dataset (16B tokens).)
- conceptnet-numberbatch-17-06-300 (size: 1225497562 bytes, description: ConceptNet Numberbatch consists of state-of-the-art semantic vectors (also known as word embeddings) that can be used directly as a representation of word meanings or as a starting point for further machine learning. ConceptNet Numberbatch is part of the ConceptNet open data project. ConceptNet provides lots of ways to compute with word meanings, one of which is word embeddings. ConceptNet Numberbatch is a snapshot of just the word embeddings. It is built using an ensemble that combines data from ConceptNet, word2vec, GloVe, and OpenSubtitles 2016, using a variation on retrofitting.)
- word2vec-ruscorpora-300 (size: 208427381 bytes, description: Word2vec Continuous Skipgram vectors trained on full Russian National 

In [17]:
def semantic_search_glove_variant():
    print("Semantic Search using GloVe")

    import gensim.downloader as api
    from sklearn.metrics.pairwise import cosine_similarity
    import numpy as np

    model = api.load("glove-wiki-gigaword-300")

    corpus = [
        "The Roman Empire was a vast and powerful civilization.",
        "Artificial intelligence is rapidly transforming various industries.",
        "The Amazon rainforest is home to incredible biodiversity.",
        "Vincent van Gogh's Starry Night is a famous masterpiece.",
        "Renewable energy sources are crucial for a sustainable future.",
        "Classical music has a rich history and profound emotional depth.",
        "Quantum computing holds the promise of revolutionary advancements."
    ]

    query = "latest technology trends"

    def avg_embedding(text):
        tokens = preprocess(text)
        word_vectors = []
        for word in tokens:
            if word in model:
                word_vectors.append(model[word])
        if len(word_vectors) == 0:
            return np.zeros(model.vector_size)
        return np.mean(word_vectors, axis=0)

    corpus_vectors = [avg_embedding(sentence) for sentence in corpus]
    query_vector = avg_embedding(query).reshape(1, -1)

    similarity_scores = []
    for i in range(len(corpus)):
        score = cosine_similarity(query_vector, corpus_vectors[i].reshape(1, -1))[0][0]
        similarity_scores.append((corpus[i], score))

    ranked = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

    print("Query:", query)
    print("\nResults:")
    for sentence, score in ranked:
        print(f"{sentence} -> {score:.4f}")

    print("\nTop Match:", ranked[0][0])

In [16]:
semantic_search_glove_variant()

Semantic Search using GloVe
Query: latest technology trends

Results:
Quantum computing holds the promise of revolutionary advancements. -> 0.6246
Renewable energy sources are crucial for a sustainable future. -> 0.5973
Artificial intelligence is rapidly transforming various industries. -> 0.5905
Classical music has a rich history and profound emotional depth. -> 0.4618
The Amazon rainforest is home to incredible biodiversity. -> 0.4253
The Roman Empire was a vast and powerful civilization. -> 0.4047
Vincent van Gogh's Starry Night is a famous masterpiece. -> 0.1996

Top Match: Quantum computing holds the promise of revolutionary advancements.
